# Carpet diagram — midHolocene temperature benchmarking

Step toward an update of IPCC AR5 WG1 Fig. 9.12 (a *portrait plot* / *carpet
diagram*). For each PMIP midHolocene (mid-Holocene (6 ka)) simulation we:

1. Load the annual-mean near-surface temperature field `tas_spatialmean_ann` from the
   CVDP output, and the matching `piControl` run.
2. Compute the **midHolocene − piControl anomaly** on each model's native grid.
3. **Sample** that anomaly field at the location of every proxy reconstruction.
4. Summarise the model–data mismatch as the **root-mean-squared error (RMSE)** against
   each reconstruction compilation (Bartlein (Bartlein et al. 2011) and Temp12k (Kaufman et al. 2020)).

Intermediate tables are written to `output/`. Run with the `my-cli-py` conda env.

In [1]:
import os, glob, re
import numpy as np
import pandas as pd
import xarray as xr

ROOT      = os.getcwd()  # run the notebook from the carpet_diagram/ directory
CVDP_DIR  = os.path.join(ROOT, 'cvdp_output_by_experiment')
EXPERIMENT = 'midHolocene'
RECON_DIR = os.path.join(ROOT, 'recons', EXPERIMENT)
OUT_DIR   = os.path.join(ROOT, 'output')
os.makedirs(OUT_DIR, exist_ok=True)

VAR        = 'tas_spatialmean_ann'
print('experiment:', EXPERIMENT)
print('CVDP   :', CVDP_DIR)
print('recons :', RECON_DIR)
print('output :', OUT_DIR)

experiment: midHolocene
CVDP   : /home/ucfaccb@ad.ucl.ac.uk/Documents/local_repos/PMIP7-vision/carpet_diagram/cvdp_output_by_experiment
recons : /home/ucfaccb@ad.ucl.ac.uk/Documents/local_repos/PMIP7-vision/carpet_diagram/recons/midHolocene
output : /home/ucfaccb@ad.ucl.ac.uk/Documents/local_repos/PMIP7-vision/carpet_diagram/output


## Step 1 — Reconstruction compilations

Two midHolocene (6 ka) compilations, each in its own CSV with a plain header row:

- **Bartlein** — the Bartlein et al. (2011) pollen-based mean-annual-temperature (MAT)
  anomalies; the annual anomaly is the `mat_anm_mean` column.
- **Temp12k** — the Temp12k database extraction (Kaufman et al. 2020; described in
  Brierley et al. 2020); the annual anomaly is the `anom` column.

Both store latitude/longitude as plain `lat`/`lon` columns (longitudes −180–180°). We
normalise them to the shared `compilation, reference, site, Proxy, Latitude, Longitude,
Anom` schema used by the rest of the pipeline.

In [2]:
# (compilation, filename, anomaly column, reference label, proxy label)
recon_specs = [
    ('Bartlein', 'Bartlein_mat.csv',       'mat_anm_mean', 'Bartlein et al. 2011',                  'pollen MAT'),
    ('Temp12k',  'temp12k_anom6k_mat.csv', 'anom',         'Temp12k (Kaufman et al. 2020)',         'multiproxy'),
]

frames = []
for comp, fname, anom_col, ref, proxy in recon_specs:
    df = pd.read_csv(os.path.join(RECON_DIR, fname))
    df.columns = [c.strip() for c in df.columns]  # some Bartlein headers carry leading spaces
    sub = pd.DataFrame({
        'compilation': comp,
        'reference': ref,
        'site': np.nan,          # these gridded compilations carry no site names
        'Proxy': proxy,
        'Latitude': df['lat'].astype(float),
        'Longitude': df['lon'].astype(float),
        'Anom': df[anom_col].astype(float),
    })
    sub['source_table'] = fname
    frames.append(sub)

recon = pd.concat(frames, ignore_index=True)
recon = recon.dropna(subset=['Latitude', 'Longitude', 'Anom'])

print(recon.groupby('compilation').size())
recon_out = os.path.join(OUT_DIR, f'recon_points_{EXPERIMENT}.csv')
recon.to_csv(recon_out, index=False)
print('wrote', recon_out)
recon.head()

compilation
Bartlein    635
Temp12k     335
dtype: int64
wrote /home/ucfaccb@ad.ucl.ac.uk/Documents/local_repos/PMIP7-vision/carpet_diagram/output/recon_points_midHolocene.csv


,compilation,reference,site,Proxy,Latitude,Longitude,Anom,source_table
0,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,29.0,-81.0,2.5390,Bartlein_mat.csv
1,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,45.0,-73.0,0.5239,Bartlein_mat.csv
2,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,45.0,-79.0,0.0033,Bartlein_mat.csv
3,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,43.0,-89.0,-0.0183,Bartlein_mat.csv
4,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,43.0,-71.0,-0.4080,Bartlein_mat.csv


## Step 2 — Model midHolocene − piControl anomalies

Each model writes its CVDP field on its own native grid, so anomalies are computed
per model. We pair every `midHolocene` file with the same model's `piControl` file
(the main file is the one named `<model>_<experiment>.cvdp_data.<years>.nc`; auxiliary
per-variable files such as `.siconc.` / `.zos.` / `.monsoon.` / `.tas.indices.` carry a
extra token before the years and are skipped). If the two grids ever differ, the control
is bilinearly regridded onto the midHolocene grid before differencing.

In [3]:
def model_files(experiment):
    """Map model name -> path for the main CVDP file of an experiment.

    Only `<model>_<experiment>.cvdp_data.<start>-<end>.nc` is the main file; the
    auxiliary per-variable outputs (`.siconc.`, `.zos.`, `.monsoon.`, `.tas.indices.`)
    put an extra token before the year range and hold no `tas_spatialmean_ann`.
    """
    out = {}
    pat = os.path.join(CVDP_DIR, experiment, f'*_{experiment}.cvdp_data.*.nc')
    main = re.compile(rf'^(?P<model>.+)_{re.escape(experiment)}\.cvdp_data\.\d+-\d+\.nc$')
    for f in sorted(glob.glob(pat)):
        m = main.match(os.path.basename(f))
        if m:
            out[m.group('model')] = f
    return out

def has_var(path):
    """True if the CVDP file actually carries the tas field (a few don't)."""
    with xr.open_dataset(path, decode_times=False) as ds:
        return VAR in ds.variables

exp_files = model_files(EXPERIMENT)
pi_files  = model_files('piControl')
paired = sorted(set(exp_files) & set(pi_files))
# Some CVDP files lack tas_spatialmean_ann entirely — drop those models.
models = [m for m in paired if has_var(exp_files[m]) and has_var(pi_files[m])]
print(f'{len(models)} models with both {EXPERIMENT} and piControl (and a tas field):')
print(models)
missing_pi = sorted(set(exp_files) - set(pi_files))
if missing_pi:
    print(f'{EXPERIMENT} models with no piControl (skipped):', missing_pi)
no_var = [m for m in paired if m not in models]
if no_var:
    print(f'models dropped — no {VAR} in CVDP file:', no_var)

32 models with both midHolocene and piControl (and a tas field):
['ACCESS-ESM1-5', 'AWI-ESM-1-1-LR', 'BCC-CSM1-1', 'CCSM4', 'CESM2', 'CNRM-CM5', 'CSIRO-Mk3-6-0', 'CSIRO-Mk3L-1-2', 'EC-Earth3', 'EC-Earth3-LR', 'FGOALS-f3-L', 'FGOALS-g2', 'FGOALS-g3', 'FGOALS-s2', 'GISS-E2-1-G', 'GISS-E2-R', 'HadGEM2-CC', 'HadGEM2-ES', 'HadGEM3-GC31-LL', 'INM-CM4-8', 'IPSL-CM5A-LR', 'IPSL-CM6A-LR', 'MIROC-ES2L', 'MIROC-ESM', 'MPI-ESM-P', 'MPI-ESM1-2-LR', 'MRI-CGCM3', 'MRI-ESM2-0', 'NESM3', 'NorESM1-F', 'NorESM2-LM', 'UofT-CCSM-4']


In [4]:
def load_field(path):
    da = xr.open_dataset(path, decode_times=False)[VAR].sortby('lat').sortby('lon')
    # A few CVDP grids (e.g. LOVECLIM piControl) carry duplicate lon values,
    # which break interpolation; keep the first occurrence of each coordinate.
    for dim in ('lat', 'lon'):
        _, idx = np.unique(da[dim].values, return_index=True)
        if len(idx) != da.sizes[dim]:
            da = da.isel({dim: np.sort(idx)})
    return da

anomalies = {}
for m in models:
    exp_field = load_field(exp_files[m])
    pi  = load_field(pi_files[m])
    if exp_field.shape != pi.shape or not (np.allclose(exp_field.lat, pi.lat) and np.allclose(exp_field.lon, pi.lon)):
        pi = pi.interp(lat=exp_field.lat, lon=exp_field.lon)
    anomalies[m] = (exp_field - pi).rename('tas_anom')
    print(f'{m:18s} grid {exp_field.shape}  mean anom {float(anomalies[m].mean()):+.2f} C')

ACCESS-ESM1-5      grid (145, 192)  mean anom -0.08 C
AWI-ESM-1-1-LR     grid (96, 192)  mean anom -0.37 C


BCC-CSM1-1         grid (64, 128)  mean anom +0.03 C


CCSM4              grid (192, 288)  mean anom -0.16 C


CESM2              grid (192, 288)  mean anom -0.10 C
CNRM-CM5           grid (128, 256)  mean anom +0.38 C


CSIRO-Mk3-6-0      grid (96, 192)  mean anom +0.15 C
CSIRO-Mk3L-1-2     grid (56, 64)  mean anom +0.13 C


...

MPI-ESM-P          grid (96, 192)  mean anom -0.12 C


MPI-ESM1-2-LR      grid (96, 192)  mean anom -0.27 C
MRI-CGCM3          grid (160, 320)  mean anom +0.15 C


MRI-ESM2-0         grid (160, 320)  mean anom -0.06 C
NESM3              grid (96, 192)  mean anom -0.17 C


NorESM1-F          grid (96, 144)  mean anom -0.32 C
NorESM2-LM         grid (96, 144)  mean anom -0.14 C


UofT-CCSM-4        grid (192, 288)  mean anom +0.01 C


## Step 3 — Sample model anomalies at reconstruction locations

Model longitudes run 0–360°, the proxy longitudes −180–180°, so targets are wrapped to
0–360 and the field is made cyclic in longitude before bilinear interpolation.

Each recon point carries a `weight` used later in the RMSE. Scattered-site compilations
weight every point equally (1.0); gridded near-global reconstructions (Cleator, Osman)
set `weight = cos(latitude)` so the RMSE is area-fair rather than pole-heavy.

In [5]:
def sample_points(field, lats, lons):
    """Bilinearly sample a (lat, lon) field at scattered points; lon made cyclic."""
    lon_cyc = np.append(field.lon.values, field.lon.values[0] + 360.0)
    fcyc = xr.concat([field, field.isel(lon=0)], dim='lon').assign_coords(lon=lon_cyc)
    ta = xr.DataArray(np.asarray(lats), dims='point')
    to = xr.DataArray(np.asarray(lons) % 360.0, dims='point')
    return fcyc.interp(lat=ta, lon=to).values

sampled = recon[['compilation', 'reference', 'site', 'Proxy',
                 'Latitude', 'Longitude', 'Anom']].copy()
sampled = sampled.rename(columns={'Anom': 'recon_anom'})
# Optional per-point weight (defaults to equal weighting when Step 1 omits it).
sampled['weight'] = recon['weight'].values if 'weight' in recon.columns else 1.0
for m in models:
    sampled[m] = sample_points(anomalies[m], sampled['Latitude'].values, sampled['Longitude'].values)

sampled_out = os.path.join(OUT_DIR, f'model_anom_at_recon_{EXPERIMENT}.csv')
sampled.to_csv(sampled_out, index=False)
print('wrote', sampled_out, '  shape', sampled.shape)
sampled.head()

wrote /home/ucfaccb@ad.ucl.ac.uk/Documents/local_repos/PMIP7-vision/carpet_diagram/output/model_anom_at_recon_midHolocene.csv   shape (970, 40)


,compilation,reference,site,Proxy,Latitude,Longitude,recon_anom,weight,ACCESS-ESM1-5,AWI-ESM-1-1-LR,...,MIROC-ES2L,MIROC-ESM,MPI-ESM-P,MPI-ESM1-2-LR,MRI-CGCM3,MRI-ESM2-0,NESM3,NorESM1-F,NorESM2-LM,UofT-CCSM-4
0,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,29.0,-81.0,2.5390,1.0,-0.242962,-0.644416,...,-0.451852,-0.262538,-0.182458,-0.287435,0.005129,-0.167620,-0.195405,-0.097478,-0.135267,-0.170893
1,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,45.0,-73.0,0.5239,1.0,-0.198058,-0.220082,...,-0.827787,-0.510614,0.008101,-0.547703,0.161884,-0.257925,-1.052231,-0.493707,-0.332427,-0.245276
2,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,45.0,-79.0,0.0033,1.0,-0.111997,-0.115241,...,-0.872071,-0.640168,0.061655,-0.506730,0.191024,-0.409908,-0.382410,-0.523424,-0.274473,-0.394154
3,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,43.0,-89.0,-0.0183,1.0,0.051769,0.536649,...,-1.103094,-0.553480,0.200128,-0.279120,0.258063,-0.254152,-1.550417,-0.577028,-0.370725,-0.513313
4,Bartlein,Bartlein et al. 2011,NaN,pollen MAT,43.0,-71.0,-0.4080,1.0,-0.181320,-0.320250,...,-0.665853,-0.292413,0.001023,-0.506090,0.104731,-0.268262,-0.711303,-0.442315,-0.412727,-0.163576


## Step 4 — RMSE of each model against each compilation

For every model × compilation we take the model-minus-proxy difference across all proxy
points in that compilation and report the (weight-weighted) RMSE and bias, plus the number
of points contributing (a point off the model grid edge can return NaN). With unit weights
this is the ordinary RMSE; gridded compilations use `cos(latitude)` weights (Step 3).

In [6]:
records = []
for m in models:
    for comp, grp in sampled.groupby('compilation'):
        diff = grp[m].values - grp['recon_anom'].values
        w = grp['weight'].values
        valid = np.isfinite(diff) & np.isfinite(w)
        n = int(valid.sum())
        if n:
            dv, wv = diff[valid], w[valid]
            rmse = float(np.sqrt(np.sum(wv * dv ** 2) / np.sum(wv)))
            bias = float(np.sum(wv * dv) / np.sum(wv))
        else:
            rmse = bias = np.nan
        records.append({'model': m, 'compilation': comp, 'n_points': n,
                        'rmse': rmse, 'bias': bias})

rmse_long = pd.DataFrame(records)
rmse_wide = rmse_long.pivot(index='model', columns='compilation', values='rmse')
rmse_wide.columns = [f'{c}_RMSE' for c in rmse_wide.columns]

rmse_long.to_csv(os.path.join(OUT_DIR, f'rmse_long_{EXPERIMENT}.csv'), index=False)
rmse_wide.to_csv(os.path.join(OUT_DIR, f'rmse_summary_{EXPERIMENT}.csv'))
print(f'wrote rmse_long_{EXPERIMENT}.csv and rmse_summary_{EXPERIMENT}.csv')
rmse_wide.sort_values(rmse_wide.columns[0])

wrote rmse_long_midHolocene.csv and rmse_summary_midHolocene.csv


,Bartlein_RMSE,Temp12k_RMSE
model,,
UofT-CCSM-4,2.347831,1.946191
ACCESS-ESM1-5,2.364555,1.970992
HadGEM3-GC31-LL,2.364591,1.917667
EC-Earth3,2.365381,1.937499
FGOALS-s2,2.367410,1.941738
BCC-CSM1-1,2.368527,1.958000
IPSL-CM5A-LR,2.373458,1.890418
CSIRO-Mk3-6-0,2.375664,1.871264
INM-CM4-8,2.377582,2.023252


These RMSE values are the building blocks of the carpet diagram: one column per model,
one row per (period, reconstruction compilation), coloured by RMSE. `carpet_figure.py`
reads every `output/rmse_long_<period>.csv`, so re-running this notebook for a new period
makes its rows appear in the portrait plot automatically.